# 码流图像分类：ByteFormer 微调 MNIST

本课程使用固定的平衡数据划分：**5,000 张训练图、1,000 张验证图、1,000 张测试图**。训练轮数与 batch size 由自己设置，batch size 可参考 32。


## 1. 检查 GPU

在 Kaggle 设置中打开 GPU 和 Internet。看到 `GPU available: True` 后继续。


In [ ]:
import sys, torch
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. 获取代码并准备资源

本 Notebook 会直接运行 Python 单元格，不需要在命令前添加 `!` 或 `%cd`。课程数据集已经放在仓库中；`prepare.py` 会检查 MNIST 文件并下载 ByteFormer 预训练权重。


In [ ]:
from pathlib import Path
import os, subprocess, sys
course_dir = Path("/kaggle/working/byteformer-mnist-course")
if not course_dir.exists():
    subprocess.run(["git", "clone", "https://github.com/Franklin-L/byteformer-mnist-course.git", str(course_dir)], check=True)
os.chdir(course_dir)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "prepare.py"], check=True)
assert Path("data/course_1of10/mnist_clean_balanced.npz").is_file()
assert Path("data/course_1of10/mnist_test_corrupted_medium.npz").is_file()
print("[READY] 代码、数据和权重已准备")


## 3. 设置参数并训练

训练集用于更新参数，验证集用于选择 `best.pt`，测试集只在模型确定后评估。`--clean-augmentations` 会在训练时使用轻微旋转与平移视图。


In [ ]:
EPOCHS = int(input("请输入训练轮数（正整数）："))
BATCH_SIZE = int(input("请输入 batch size（参考32，正整数）："))
assert EPOCHS > 0 and BATCH_SIZE > 0
subprocess.run([sys.executable, "train_course_subset.py", "--method", "clean", "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE), "--clean-augmentations", "--output", "outputs/course_clean"], check=True)


## 4. 查看学习曲线与验证结果

观察训练损失是否总体下降、验证准确率如何变化，并记录最佳轮次。


In [ ]:
import json
from IPython.display import display, Image as DisplayImage
baseline_dir = Path("outputs/course_clean")
metrics = json.loads((baseline_dir / "metrics.json").read_text())
print("最佳轮次:", metrics["best_epoch"])
print("最佳验证选择分数:", f'{metrics["best_selection_score"]:.2%}')
print("训练耗时:", f'{metrics["elapsed_seconds"]:.1f} 秒')
display(DisplayImage(filename=str(baseline_dir / "curves.png")))


## 5. 在独立测试集上评估

基础任务只评估干净测试集，不运行损坏测试。损坏测试集只在后面的加分项中使用。


In [ ]:
subprocess.run([sys.executable, "evaluate.py", "--checkpoint", "outputs/course_clean/best.pt", "--output", "outputs/course_clean_eval"], check=True)
evaluation = json.loads(Path("outputs/course_clean_eval/evaluation.json").read_text())
result = evaluation["test"]
print(f'Clean: {result["accuracy"]:.2%} ({result["correct"]}/{result["samples"]})')
display(DisplayImage(filename="outputs/course_clean_eval/predictions.png"))


## 6. 单图预测

`--index` 使用官方 MNIST 测试索引，可改为 0–9999 中的其他数字。


In [ ]:
subprocess.run([sys.executable, "predict.py", "--checkpoint", "outputs/course_clean/best.pt", "--index", "0"], check=True)
display(DisplayImage(filename="outputs/course_clean/prediction_single.png"))


## 7. 参数对比

自行设置第二组轮数与 batch size。建议一次只改变一个参数，其他条件保持一致。


In [ ]:
COMPARISON_EPOCHS = int(input("请输入对比实验轮数："))
COMPARISON_BATCH_SIZE = int(input("请输入对比 batch size："))
assert COMPARISON_EPOCHS > 0 and COMPARISON_BATCH_SIZE > 0
subprocess.run([sys.executable, "train_course_subset.py", "--method", "clean", "--epochs", str(COMPARISON_EPOCHS), "--batch-size", str(COMPARISON_BATCH_SIZE), "--clean-augmentations", "--output", "outputs/comparison"], check=True)
comparison = json.loads(Path("outputs/comparison/metrics.json").read_text())
print("第一次:", metrics["config"]["epochs"], metrics["config"]["batch_size"], f'{metrics["best_selection_score"]:.2%}')
print("对比实验:", comparison["config"]["epochs"], comparison["config"]["batch_size"], f'{comparison["best_selection_score"]:.2%}')


## 8. 加分项：损坏增强

- **Bit flip**：选中一个字节，随机翻转其中一位，码流长度不变。
- **Byte loss**：删除选中的字节，后续字节前移。

完整说明见 `docs/bonus_corruption.md`。加分项训练时加入随机损坏码流，训练完成后再评估 Clean、Medium-Flip、Medium-Loss 三类测试结果。参考资料见 `research/04_bonus_corrupted_bitstream_references.md`。


In [ ]:
# 加分项：损坏增强（选做）
# 本 Notebook 不提供现成的增强训练答案。
# 请打开 examples/bonus_augmentation_template.py，根据伪代码完成自己的训练脚本。
# 完成并保存 best.pt 后，再运行：
# python evaluate_course_corruption.py \
#   --checkpoint <你的加分模型>/best.pt \
#   --output <你的加分模型>_eval


## 9. 下载结果

将 JSON、CSV、PNG 和预测数组打包。大型模型文件不放入压缩包。


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import FileLink
archive = Path("/kaggle/working/byteformer_mnist_results.zip")
files = [p for p in Path("outputs").rglob("*") if p.is_file() and p.suffix in {".json", ".csv", ".png", ".npz"}]
with ZipFile(archive, "w", ZIP_DEFLATED) as zf:
    for path in files:
        zf.write(path, path.as_posix())
print("已打包:", archive)
FileLink(str(archive))


## 提交前检查

- 已记录训练轮数、batch size、运行环境和耗时。
- 已区分训练集、验证集和测试集的用途。
- 已保存学习曲线、测试结果、单图预测和参数对比。
- 加分实验若未完成，不影响基础任务提交。
